In [ ]:
# !pip install ipython==8.1.0

In [ ]:
# # Comment the following if you are running your code locally
# # This mounts your Google Drive to the Colab VM.
# from google.colab import drive
# drive.mount('/content/drive')
# drive_folder = '/content/drive/MyDrive'
# notebook_folder = drive_folder + '/project'
# %cd {notebook_folder}

# %cd ./cs682/data
# !bash prepare.sh
# %cd ../../

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from transformers import get_linear_schedule_with_warmup
import json

from cs682.models.teacher import BERTTeacher
from cs682.models.student import FunnelTransformer
from cs682.data.loader import IMDBDataset, YelpDataset, AmazonDataset
from cs682.evaluator import evaluate

In [ ]:
class StudentTrainConfig:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

    def __repr__(self):
        return f"StudentTrainConfig({self.__dict__})"

def distillation_loss(
    student_out: dict,
    teacher_logits: torch.Tensor,
    teacher_cls: list[torch.Tensor],
    labels: torch.Tensor,
    alpha: float = 1.0,
    beta: float = 1.0,
    gamma: float = 1.0,
    temperature: float = 4.0,
) -> dict[str, torch.Tensor]:
    """
    Combined distillation objective:
        L = alpha * L_task  +  beta * L_logit  +  gamma * L_layer

    Args:
        student_out:    dict from FunnelTransformer.forward()
        teacher_logits: (B, C) teacher final logits
        teacher_cls:    list of 3 tensors (B, D) — teacher CLS at mapped layers
        labels:         (B,) ground-truth class indices
        temperature:    softmax temperature for KL divergence
    """
    s_logits = student_out["logits"]
    s_cls_list = student_out["cls_hiddens"]

    l_task = F.cross_entropy(s_logits, labels)

    s_soft = F.log_softmax(s_logits / temperature, dim=-1)
    t_soft = F.softmax(teacher_logits / temperature, dim=-1)
    l_logit = F.kl_div(s_soft, t_soft, reduction="batchmean") * (temperature**2)

    if len(s_cls_list) != len(teacher_cls):
        raise ValueError(
            f"Expected {len(s_cls_list)} teacher CLS tensors "
            f"(one per block), got {len(teacher_cls)}"
        )
    l_layer = sum(
        F.mse_loss(s_cls, t_cls) for s_cls, t_cls in zip(s_cls_list, teacher_cls)
    ) / len(s_cls_list)

    loss = alpha * l_task + beta * l_logit + gamma * l_layer

    return {
        "loss": loss,
        "l_task": l_task.detach(),
        "l_logit": l_logit.detach(),
        "l_layer": l_layer.detach(),
    }
